# 3장 2강: SDK 개발 환경 구성과 연결 테스트
## 3. Supabase Client 초기화

In [8]:
import os
from supabase import create_client

url = os.getenv("SUPABASE_URL")
key = os.getenv("SUPABASE_PUBLISHABLE_KEY")

# Client 초기화
supabase = create_client(url,key)

print(supabase)

## 4. 연결 테스트와 응답·오류 객체 확인

# 3장 3강: SDK 기반 데이터 생성 구현

## 1. SDK insert 구조

In [27]:
# 사용자 추가
response = supabase.table("users").insert({"email": "user03@test.org"}).execute()

response.data

[{'id': 'ae33bef5-1641-4197-ae76-1dfb7d25894d',
  'email': 'user03@test.org',
  'created_at': '2026-09-09T02:37:56.801086'}]

In [31]:
# documents에 추가
response = supabase.table("documents").insert({
    "title": "두 번째 문서",
    "content": "본문 내용",
    "user_id" :"ae33bef5-1641-4197-ae76-1dfb7d25894d"
}).execute()

response.data

[{'id': '4f0cbd58-094c-4a30-b97f-26f71035f1ac',
  'title': '두 번째 문서',
  'content': '본문 내용',
  'created_at': '2026-09-09T02:39:11.624089',
  'user_id': 'ae33bef5-1641-4197-ae76-1dfb7d25894d'}]

In [17]:
response = supabase.table("users").select("*").execute()

response.data

[{'id': 'e25690b3-e5b2-4778-86b9-29b7b158df2b',
  'email': 'user02@test.org',
  'created_at': '2026-09-09T00:42:49.513193'},
 {'id': '724bab23-484e-46ab-a801-f206f370fdc7',
  'email': 'user01@test.org',
  'created_at': '2026-09-09T00:52:46.054893'},
 {'id': '187b2319-fade-483b-8136-20c9d52ceee0',
  'email': 'user03@test.org',
  'created_at': '2026-09-09T02:16:31.01348'}]

In [ ]:
response = (
    supabase.table("users")
        .select("*")
        .ilike("email","user02%")
        .execute()
)

id, email, created_at = response.data[0]

print(f"id:{id}, email: {email}, created_at: {created_at}")

id:id, email: email, created_at: created_at


In [ ]:
response = (
    supabase.table("users")
        .select("*")
        .ilike("email","user04%")
        .single() # 레코드1개, 무조건 1개가 나와야 한다, 아니면 예외발생, 반환값은 딕셔너리
        .execute()
)

id, email, created_at = response.data[0]

print(f"id:{id}, email: {email}, created_at: {created_at}")

In [24]:
response = (
    supabase.table("users")
        .select("*")
        .ilike("email","user03%")
        .limit(1)
        .offset(1) # 1 인덱스에서 1개를 조회
        .maybe_single() # 0~1, 0개이면 None
        .execute()
)

print(response)

None


In [25]:
response = (
    supabase.table("users")
        .select("*")
        .order("email", desc=True)
        .execute()
)

response.data

[{'id': '187b2319-fade-483b-8136-20c9d52ceee0',
  'email': 'user03@test.org',
  'created_at': '2026-09-09T02:16:31.01348'},
 {'id': 'e25690b3-e5b2-4778-86b9-29b7b158df2b',
  'email': 'user02@test.org',
  'created_at': '2026-09-09T00:42:49.513193'},
 {'id': '724bab23-484e-46ab-a801-f206f370fdc7',
  'email': 'user01@test.org',
  'created_at': '2026-09-09T00:52:46.054893'}]

In [32]:
# LEFT JOIN

response = (
    supabase.table("documents")
        .select("id, title, content, users(id, email)")
        .execute()
)

response.data

[{'id': 'bd8d0c04-10e5-48f2-b154-0287b895702e',
  'title': '첫 번째 문서',
  'content': '본문 내용',
  'users': {'id': 'ae33bef5-1641-4197-ae76-1dfb7d25894d',
   'email': 'user03@test.org'}},
 {'id': '4f0cbd58-094c-4a30-b97f-26f71035f1ac',
  'title': '두 번째 문서',
  'content': '본문 내용',
  'users': {'id': 'ae33bef5-1641-4197-ae76-1dfb7d25894d',
   'email': 'user03@test.org'}}]

In [ ]:
# INNER JOIN
response = (
    supabase.table("documents")
        .select("id,title,content,users!inner(id,email)")
        .ilike("users.email","user%")
        .execute()
)

response.data

[{'id': 'bd8d0c04-10e5-48f2-b154-0287b895702e',
  'title': '첫 번째 문서',
  'content': '본문 내용',
  'users': {'id': 'ae33bef5-1641-4197-ae76-1dfb7d25894d',
   'email': 'user03@test.org'}},
 {'id': '4f0cbd58-094c-4a30-b97f-26f71035f1ac',
  'title': '두 번째 문서',
  'content': '본문 내용',
  'users': {'id': 'ae33bef5-1641-4197-ae76-1dfb7d25894d',
   'email': 'user03@test.org'}}]

## 2. 입력값과 DB 컬럼 매핑

## 3. 생성 결과와 오류 응답 처리

In [ ]:
create_document(None, "본문 내용입니다.", 1)

# 3장 4강: SDK 기반 데이터 조회 구현

## 1. SDK select 구조

## 2. 조건 필터링: eq, order, limit

## 3. 사용자별 조회와 결과 리스트 처리

# 3장 5강: SDK 기반 데이터 수정·삭제 구현

## 1. SDK update 구조

In [37]:
response = (
    supabase.table("documents")
        .select("*")
        .execute()
)

print(response.data)

[{'id': '4f0cbd58-094c-4a30-b97f-26f71035f1ac', 'title': '두 번째 문서', 'content': '본문 내용', 'created_at': '2026-09-09T02:39:11.624089', 'user_id': 'ae33bef5-1641-4197-ae76-1dfb7d25894d'}, {'id': 'bd8d0c04-10e5-48f2-b154-0287b895702e', 'title': '첫 번째 문서(수정)', 'content': '본문 내용', 'created_at': '2026-09-09T02:39:05.531568', 'user_id': 'ae33bef5-1641-4197-ae76-1dfb7d25894d'}]


In [38]:
response = (
    supabase.table("documents")
        .update({
            "title": "첫 번째 문서(수정)"
        })
        .eq("id", "bd8d0c04-10e5-48f2-b154-0287b895702e")
        .execute()
)
response.data

[{'id': 'bd8d0c04-10e5-48f2-b154-0287b895702e',
  'title': '첫 번째 문서(수정)',
  'content': '본문 내용',
  'created_at': '2026-09-09T02:39:05.531568',
  'user_id': 'ae33bef5-1641-4197-ae76-1dfb7d25894d'}]

## 2. SDK delete 구조

In [41]:
response = (
    supabase.table("users")
        .select("*")
        .execute()

)

response.data

[{'id': 'e25690b3-e5b2-4778-86b9-29b7b158df2b',
  'email': 'user02@test.org',
  'created_at': '2026-09-09T00:42:49.513193'},
 {'id': '187b2319-fade-483b-8136-20c9d52ceee0',
  'email': 'user03@test.org',
  'created_at': '2026-09-09T02:16:31.01348'},
 {'id': '5f7616a0-f8d9-444e-b4ef-2f951f45e4ee',
  'email': 'user03@test.org',
  'created_at': '2026-09-09T02:37:49.353657'},
 {'id': 'ae33bef5-1641-4197-ae76-1dfb7d25894d',
  'email': 'user03@test.org',
  'created_at': '2026-09-09T02:37:56.801086'}]

In [40]:
response = (
    supabase.table("users")
        .delete()
        .eq("id","724bab23-484e-46ab-a801-f206f370fdc7")
        .execute()
)

response.data

[{'id': '724bab23-484e-46ab-a801-f206f370fdc7',
  'email': 'user01@test.org',
  'created_at': '2026-09-09T00:52:46.054893'}]

## 3. 수정·삭제 후 검증과 오류 처리

In [ ]:
# 검증: 상태가 잘 바뀌었는지 다시 조회


# 3장 6강: Supabase Authentication 기본 흐름

## 2. 이메일 기반 회원가입과 로그인

In [ ]:
key = os.getenv("SUPABASE_ANON_KEY")
supabase = create_client(url,key)

"""
supabase.auth : 인증(로그인) / 인가(접근 제한) 관련 메서드 가지고 있는 객체

테이블명: auth.users
"""

# 회원가입

supabase.auth.sign_up({
    "email":"user01@test.org",
    "password": "password1234",
    "phone": "01010001000"
})

response

APIResponse(data=[{'id': 'e25690b3-e5b2-4778-86b9-29b7b158df2b', 'email': 'user02@test.org', 'created_at': '2026-09-09T00:42:49.513193'}, {'id': '187b2319-fade-483b-8136-20c9d52ceee0', 'email': 'user03@test.org', 'created_at': '2026-09-09T02:16:31.01348'}, {'id': '5f7616a0-f8d9-444e-b4ef-2f951f45e4ee', 'email': 'user03@test.org', 'created_at': '2026-09-09T02:37:49.353657'}, {'id': 'ae33bef5-1641-4197-ae76-1dfb7d25894d', 'email': 'user03@test.org', 'created_at': '2026-09-09T02:37:56.801086'}], count=None)

In [45]:
# 로그인
response = supabase.auth.sign_in_with_password({
    "email": "user01@test.org",
    "password": "password1234"
})

response.user

User(id='b43d3b6c-e40b-41f1-b62c-ef6424594f96', app_metadata={'provider': 'email', 'providers': ['email']}, user_metadata={'email': 'user01@test.org', 'email_verified': True, 'phone_verified': False, 'sub': 'b43d3b6c-e40b-41f1-b62c-ef6424594f96'}, aud='authenticated', confirmation_sent_at=None, recovery_sent_at=None, email_change_sent_at=None, new_email=None, new_phone=None, invited_at=None, action_link=None, email='user01@test.org', phone='', created_at=datetime.datetime(2026, 9, 9, 3, 40, 6, 724772, tzinfo=TzInfo(0)), confirmed_at=datetime.datetime(2026, 9, 9, 3, 40, 6, 742012, tzinfo=TzInfo(0)), email_confirmed_at=datetime.datetime(2026, 9, 9, 3, 40, 6, 742012, tzinfo=TzInfo(0)), phone_confirmed_at=None, last_sign_in_at=datetime.datetime(2026, 9, 9, 3, 48, 22, 711954, tzinfo=TzInfo(0)), role='authenticated', updated_at=datetime.datetime(2026, 9, 9, 3, 48, 22, 724348, tzinfo=TzInfo(0)), identities=[UserIdentity(id='b43d3b6c-e40b-41f1-b62c-ef6424594f96', identity_id='e128cc71-774d-48e

## 3. 세션, 로그아웃, user 객체

In [48]:
# 현재 로그인된 사용자 확인
user = supabase.auth.get_user() # 현재 로그인한 사용자 정보, 있으면 로그인 상태, None이면 미 인증 상태
if user: # 로그인 상태
    print("로그인 상태:", user)
else:
    print("미 로그인 상태")

미 로그인 상태


In [47]:
# 로그아웃
supabase.auth.sign_out()

## 4. 사용자 ID와 테이블 데이터 연결

# 3장 7강: 사용자별 데이터 접근과 권한 관리

## 2. user_id 기반 필터링

In [ ]:


# 본인 문서만 조회


# 3장 8강: SDK 기반 미니 과제 - 사용자별 문서 CRUD

## 2. Auth와 user_id 연결

In [ ]:
import os
from supabase import create_client
url = os.getenv("SUPABASE_URL")
key = os.getenv("SUPABASE_ANON_KEY")

supabase = create_client(url,key)

supabase.auth.sign_in_with_password({
    "email": "user01@test.org",
    "password": "password1234"
})

user = supabase.auth.get_user().user
user_id = user.id

# 생성: 로그인 사용자의 문서
response = (
    supabase.table("documents")
        .insert({
            "title": "첫번째 문서",
            "content": "본문 내용",
            "user_id": user_id
        })
        .execute()
)
print(response)

# 조회: 본인 문서만


data=[{'id': '31c9c36b-a70b-4884-96a8-2c7cbb60f143', 'title': '첫번째 문서', 'content': '본문 내용', 'created_at': '2026-09-10T01:09:27.429772', 'user_id': 'b43d3b6c-e40b-41f1-b62c-ef6424594f96'}] count=None


In [18]:
# CRUD 클래스
class AiLog:
    def __init__(self):
        user = supabase.auth.get_user()
        if not user:
            raise PermissionError("로그인이 필요합니다.")

        self.user = user.user

    def create_documents(self, title, content):

        response = (
            supabase.table("documents")
                .insert({
                    "title": title,
                    "content": content,
                    "user_id": self.user.id
                })
                .execute()
        )

        return response.data

    def get_my_documents(self):
        response = (
            supabase.table("documents")
                .select("*")
                .eq("user_id", self.user.id)
                .execute()
        )

        return response.data

In [11]:
ai_log = AiLog()

ai_log.create_documents("세 번째 문서", "본문 내용")

[{'id': '01fb9caf-b31a-4b45-8047-cf292a0c40a0',
  'title': '세 번째 문서',
  'content': '본문 내용',
  'created_at': '2026-09-10T01:27:27.38993',
  'user_id': 'b43d3b6c-e40b-41f1-b62c-ef6424594f96'}]

In [19]:
ai_log = AiLog()

ai_log.get_my_documents()

[{'id': '31c9c36b-a70b-4884-96a8-2c7cbb60f143',
  'title': '첫번째 문서',
  'content': '본문 내용',
  'created_at': '2026-09-10T01:09:27.429772',
  'user_id': 'b43d3b6c-e40b-41f1-b62c-ef6424594f96'},
 {'id': '0b75cb72-2aff-47cd-a259-054656da84e5',
  'title': '두 번째 문서',
  'content': '본문 내용',
  'created_at': '2026-09-10T01:21:00.911735',
  'user_id': 'b43d3b6c-e40b-41f1-b62c-ef6424594f96'},
 {'id': 'e84368a2-f834-4d56-928b-dc7fdbeb1843',
  'title': '세 번째 문서',
  'content': '본문 내용',
  'created_at': '2026-09-10T01:26:54.371311',
  'user_id': 'b43d3b6c-e40b-41f1-b62c-ef6424594f96'},
 {'id': '01fb9caf-b31a-4b45-8047-cf292a0c40a0',
  'title': '세 번째 문서',
  'content': '본문 내용',
  'created_at': '2026-09-10T01:27:27.38993',
  'user_id': 'b43d3b6c-e40b-41f1-b62c-ef6424594f96'}]

In [20]:
supabase.auth.sign_up({
    "email": "user02@test.org",
    "password": "password1234"
})

AuthResponse(user=User(id='47e35d0d-0970-40c8-9778-34a18926f8bb', app_metadata={'provider': 'email', 'providers': ['email']}, user_metadata={'email': 'user02@test.org', 'email_verified': True, 'phone_verified': False, 'sub': '47e35d0d-0970-40c8-9778-34a18926f8bb'}, aud='authenticated', confirmation_sent_at=None, recovery_sent_at=None, email_change_sent_at=None, new_email=None, new_phone=None, invited_at=None, action_link=None, email='user02@test.org', phone='', created_at=datetime.datetime(2026, 9, 10, 2, 12, 32, 981909, tzinfo=TzInfo(0)), confirmed_at=None, email_confirmed_at=datetime.datetime(2026, 9, 10, 2, 12, 33, 21344, tzinfo=TzInfo(0)), phone_confirmed_at=None, last_sign_in_at=datetime.datetime(2026, 9, 10, 2, 12, 33, 25645, tzinfo=TzInfo(0)), role='authenticated', updated_at=datetime.datetime(2026, 9, 10, 2, 12, 33, 31914, tzinfo=TzInfo(0)), identities=[UserIdentity(id='47e35d0d-0970-40c8-9778-34a18926f8bb', identity_id='02a91953-3f05-4857-9c67-1f740cfff1bb', user_id='47e35d0d-

In [22]:
supabase.auth.sign_in_with_password({
    "email": "user02@test.org",
    "password": "password1234"
})

AuthResponse(user=User(id='47e35d0d-0970-40c8-9778-34a18926f8bb', app_metadata={'provider': 'email', 'providers': ['email']}, user_metadata={'email': 'user02@test.org', 'email_verified': True, 'phone_verified': False, 'sub': '47e35d0d-0970-40c8-9778-34a18926f8bb'}, aud='authenticated', confirmation_sent_at=None, recovery_sent_at=None, email_change_sent_at=None, new_email=None, new_phone=None, invited_at=None, action_link=None, email='user02@test.org', phone='', created_at=datetime.datetime(2026, 9, 10, 2, 12, 32, 981909, tzinfo=TzInfo(0)), confirmed_at=datetime.datetime(2026, 9, 10, 2, 12, 33, 21344, tzinfo=TzInfo(0)), email_confirmed_at=datetime.datetime(2026, 9, 10, 2, 12, 33, 21344, tzinfo=TzInfo(0)), phone_confirmed_at=None, last_sign_in_at=datetime.datetime(2026, 9, 10, 2, 14, 5, 437507, tzinfo=TzInfo(0)), role='authenticated', updated_at=datetime.datetime(2026, 9, 10, 2, 14, 5, 447568, tzinfo=TzInfo(0)), identities=[UserIdentity(id='47e35d0d-0970-40c8-9778-34a18926f8bb', identity

In [ ]:
response = (
    supabase.table("documents")
        .select("*")
        .execute()
)

response.data

## 3. 응답·오류 처리와 코드 구조 정리